In [1]:
#import statements
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import matplotlib.pyplot as plt
from torchinfo import summary
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder
import nilearn.image
import nilearn.plotting
import copy
from torch.utils.data import random_split, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, r2_score
from sklearn.preprocessing import label_binarize
from pathlib import Path
from scipy import signal
import mne
from mne.preprocessing import ICA
from mne_icalabel.iclabel import iclabel_label_components

/Users/william.wakefield/PycharmProjects/Mayo_EEG_FDG_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
mne.set_log_level('WARNING')
INPUT_DIR = Path("model_data/orig_eeg_raw")
OUTPUT_DIR = Path("model_data/non_ica")

## Non-ICA pre-processing

In [4]:
def process_subject_non_ica(df):
    start_ids = np.sort(df['start_index'].unique())
    first_block = df[df['start_index'] == start_ids[0]]
    channel_names = first_block['channel_name'].astype(str).tolist()
    dupes = [c for c in set(channel_names) if channel_names.count(c) > 1]
    if dupes:
        raise ValueError(
            f"Duplicate channel names {sorted(dupes)} within start_index "
            f"{start_ids[0]} — parquet contains overlapping recordings."
        )

    seg_arrays, seg_lengths = [], []
    for sid in start_ids:
        block = df[df['start_index'] == sid]
        block_chs = block['channel_name'].astype(str).tolist()
        if len(block_chs) != len(channel_names) or set(block_chs) != set(channel_names):
            raise ValueError(
                f"start_index {sid} has channels {block_chs}, "
                f"expected {channel_names}."
            )
        seg_data = np.stack([
            np.asarray(
                block.loc[block['channel_name'].astype(str) == ch, 'segment'].values[0],
                dtype=np.float64,
            )
            for ch in channel_names
        ])
        seg_arrays.append(seg_data)
        seg_lengths.append(seg_data.shape[1])
    data = np.concatenate(seg_arrays, axis=1)  # (n_channels, n_total_samples)

    info = mne.create_info(ch_names=channel_names, sfreq=256, ch_types='eeg')
    raw = mne.io.RawArray(data, info)
    raw.filter(l_freq=0.5, h_freq=45, method='fir',
               phase='zero', fir_window='hamming')
    raw.set_eeg_reference('average', projection=False)

    arr = raw.get_data()
    ch_mean = arr.mean(axis=1, keepdims=True)
    ch_std = arr.std(axis=1, keepdims=True) + 1e-8
    arr = (arr - ch_mean) / ch_std

    out_blocks = []
    offset = 0
    for new_seg_idx, (sid, slen) in enumerate(zip(start_ids, seg_lengths)):
        seg_clean = arr[:, offset:offset + slen]
        offset += slen
        block = df[df['start_index'] == sid].copy()
        block['segment_index'] = new_seg_idx  # renumber to 0..N-1
        for ch_idx, ch in enumerate(channel_names):
            row_idx = block.index[block['channel_name'].astype(str) == ch][0]
            block.at[row_idx, 'segment'] = seg_clean[ch_idx].astype(np.float32)
        out_blocks.append(block)
    return pd.concat(out_blocks, ignore_index=True)

In [5]:
skipped = []
for parquet_path in sorted(INPUT_DIR.glob("*.parquet")):
    subject_id = parquet_path.stem
    print(f"Processing {subject_id} ... ", end="", flush=True)
    try:
        df = pd.read_parquet(parquet_path)
        df_out = process_subject_non_ica(df)
        df_out.to_parquet(OUTPUT_DIR / parquet_path.name, index=False)
        print("ok")
    except Exception as e:
        print(f"SKIPPED — {e}")
        skipped.append((subject_id, str(e)))

Processing 100584250 ... ok
Processing 100597363 ... ok
Processing 100864302 ... ok
Processing 101382732 ... ok
Processing 102841616 ... ok
Processing 102933094 ... ok
Processing 103293424 ... ok
Processing 104115732 ... ok
Processing 104388202 ... ok
Processing 104454780 ... ok
Processing 104794142 ... ok
Processing 104921540 ... ok
Processing 106169522 ... ok
Processing 106714847 ... ok
Processing 107333864 ... ok
Processing 107501813 ... ok
Processing 107943342 ... ok
Processing 107969830 ... ok
Processing 107992048 ... ok
Processing 108540365 ... ok
Processing 108853799 ... ok
Processing 109372236 ... ok
Processing 109400346 ... ok
Processing 109711792 ... ok
Processing 109801129 ... ok
Processing 109942570 ... ok
Processing 110034837 ... ok
Processing 110717532 ... ok
Processing 110770774 ... ok
Processing 110816623 ... ok
Processing 110871979 ... ok
Processing 111398473 ... ok
Processing 111452972 ... ok
Processing 111886762 ... ok
Processing 111991802 ... ok
Processing 112490764

In [6]:
pca_vals = pd.read_parquet("model_data/matched_pca_vectors.parquet")

### ICA

In [4]:
ICA_OUTPUT_DIR = Path("model_data/post_ica")

N_COMPONENTS  = 15      # per subject; clamped to n_channels - 1 automatically
LABEL_THRESH  = 0.70    # confidence to call a component non-brain
ARTIFACT_TYPES = {1, 2, 3, 4}  # muscle, eye, heart, line_noise  (index 0 = brain)

In [ ]:
def process_subject_ica(df):
    start_ids = np.sort(df['start_index'].unique())
    channel_names = df[df['start_index'] == start_ids[0]]['channel_name'].astype(str).tolist()

    seg_arrays, seg_lengths = [], []
    for sid in start_ids:
        block = df[df['start_index'] == sid]
        seg_data = np.stack([
            np.asarray(block.loc[block['channel_name'].astype(str) == ch,
                                 'segment'].values[0], dtype=np.float64)
            for ch in channel_names
        ])
        seg_arrays.append(seg_data)
        seg_lengths.append(seg_data.shape[1])
    data = np.concatenate(seg_arrays, axis=1)

    info = mne.create_info(ch_names=channel_names, sfreq=256, ch_types='eeg')
    raw  = mne.io.RawArray(data, info)
    montage = mne.channels.make_standard_montage('standard_1020')
    raw.set_montage(montage, on_missing='ignore')
    raw.set_eeg_reference('average')

    # ── 3. High-pass copy for ICA fitting (1 Hz prevents slow-drift issues) ─
    raw_hpf = raw.copy().filter(l_freq=1.0, h_freq=45.0)

    # ── 4. Fit ICA ──────────────────────────────────────────────────────────
    n_comp = min(N_COMPONENTS, len(channel_names) - 1)
    ica = ICA(n_components=n_comp, method='fastica', random_state=42, max_iter=800)
    ica.fit(raw_hpf)

    # ── 5. Auto-label with ICLabel; exclude artifact components ────────────
    try:
        labels = iclabel_label_components(raw_hpf, ica)   # shape (n_comp, 7)
        ica.exclude = [
            i for i, probs in enumerate(labels['y_pred_proba'])
            if np.argmax(probs) in ARTIFACT_TYPES and np.max(probs) >= LABEL_THRESH
        ]
    except Exception:
        ica.exclude = []   # no montage match — skip exclusion, keep raw ICA

    # ── 6. Apply to broadband-filtered signal ─────────────────────────────
    raw.filter(l_freq=0.5, h_freq=45.0,
               method='fir', phase='zero', fir_window='hamming')
    ica.apply(raw)

    # ── 7. Z-score normalize ───────────────────────────────────────────────
    arr = raw.get_data()
    arr = (arr - arr.mean(axis=1, keepdims=True)) / (arr.std(axis=1, keepdims=True) + 1e-8)

    # ── 8. Re-split into original segments ────────────────────────────────
    out_blocks, offset = [], 0
    for new_idx, (sid, slen) in enumerate(zip(start_ids, seg_lengths)):
        seg_clean = arr[:, offset:offset + slen]
        offset += slen
        block = df[df['start_index'] == sid].copy()
        block['segment_index'] = new_idx
        for ch_idx, ch in enumerate(channel_names):
            row = block.index[block['channel_name'].astype(str) == ch][0]
            block.at[row, 'segment'] = seg_clean[ch_idx].astype(np.float32)
        out_blocks.append(block)
    return pd.concat(out_blocks, ignore_index=True)

In [ ]:
skipped_ica = []
for parquet_path in sorted(INPUT_DIR.glob("*.parquet")):
    subject_id = parquet_path.stem
    print(f"Processing {subject_id} ... ", end="", flush=True)
    try:
        df = pd.read_parquet(parquet_path)
        df_out = process_subject_ica(df)
        df_out.to_parquet(ICA_OUTPUT_DIR / parquet_path.name, index=False)
        print(f"ok  (excluded {len(ica.exclude)} components)")
    except Exception as e:
        print(f"SKIPPED — {e}")
        skipped_ica.append((subject_id, str(e)))